# Exploration CANN — Module Pricing

**⚠️ Notebook d'archive — NE PAS RELANCER.**

Ce notebook conserve la trace complète de la démarche exploratoire ayant mené aux résultats retenus dans le rapport (chapitre 4) : débogage numérique du CANN générique, tests d'interactions ciblées (paire `DrivAge x BonusMalus`, groupe `VehPower/VehAge/VehGas/VehBrand`), diagnostics intermédiaires (gradient, résidus).

Pour reproduire uniquement les résultats finaux rapidement, utiliser `03b_pricing_cann_final.ipynb`, qui charge les modèles déjà entraînés depuis `models/`.

## Setup

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device utilisé :", device)

## Chargement des données et baseline GLM

In [ ]:
from src.pricing.data import build_pricing_dataset, train_valid_test_split, get_severity_subset
from src.pricing.features import build_features
from src.pricing.models import fit_glm_poisson, fit_glm_gamma, predict_frequency, predict_severity

df = build_pricing_dataset()
df = build_features(df)
train, valid, test = train_valid_test_split(df)

model_glm_train = fit_glm_poisson(train)
print("Déviance/obs train :", model_glm_train.deviance / len(train))

train_sev = get_severity_subset(train)
test_sev = get_severity_subset(test)
model_gamma = fit_glm_gamma(train_sev)

## Préparation des prédictions GLM pour le CANN

`glm_log_pred` sert de skip connection fixe. `log_mu_glm` (= `glm_log_pred + log(Exposure)`) sert de working weight pour les modèles d'interaction ciblée (section suivante).

In [ ]:
train_freq_pred = predict_frequency(model_glm_train, train)
valid_freq_pred = predict_frequency(model_glm_train, valid)
test_freq_pred = predict_frequency(model_glm_train, test)

train["glm_log_pred"] = np.log(train_freq_pred)
valid["glm_log_pred"] = np.log(valid_freq_pred)
test["glm_log_pred"] = np.log(test_freq_pred)

train["log_mu_glm"] = train["glm_log_pred"] + np.log(train["Exposure"])
valid["log_mu_glm"] = valid["glm_log_pred"] + np.log(valid["Exposure"])
test["log_mu_glm"] = test["glm_log_pred"] + np.log(test["Exposure"])

## Tentative 1 — CANN générique (toutes variables)

**Résultat retenu dans le rapport : meilleur valid_loss 0.3077-0.3078, n'améliore pas le GLM (0.3072).**

Débogage numérique effectué avant d'obtenir ce résultat stable :
- `NaN` initiaux dus à `log(0)` dans la déviance de Poisson → corrigé par substitution sécurisée (`torch.where` avec valeur de repli)
- `lr=1e-3` initial trop instable sans gradient clipping → stabilisé avec `clip_grad_norm_(max_norm=1.0)`
- Vérification de l'initialisation à zéro (CANN = GLM exact à l'epoch 0, écart 0.0)

In [ ]:
from src.pricing.cann import (
    FreMTPL2Dataset, CANNFrequencyNet, CATEGORICAL_CARDINALITIES, train_cann
)

train_dataset = FreMTPL2Dataset(train)
valid_dataset = FreMTPL2Dataset(valid)

model_cann_generic = CANNFrequencyNet(n_continuous=5, categorical_cardinalities=CATEGORICAL_CARDINALITIES)
model_cann_generic, history = train_cann(
    model_cann_generic, train_dataset, valid_dataset, n_epochs=100, lr=1e-3, device=device
)
# Meilleur résultat observé : valid_loss ~0.3077-0.3078 (epoch ~55), au-dessus du GLM (0.3072)

## Tentative 2 — Interaction ciblée : DrivAge x BonusMalus

**Résultat retenu : meilleur valid_loss 0.3093, n'améliore pas le GLM.** Interaction identifiée comme significative par Schelldorfer & Wüthrich (2019) sur leur portefeuille, non reproduite ici — hypothèse : différences de nettoyage de données ou de split.

In [ ]:
from src.pricing.cann import PairInteractionNet, PairDataset, poisson_deviance_loss_v2

train_pair = PairDataset(train, "DrivAge_norm", "BonusMalus_norm")
valid_pair = PairDataset(valid, "DrivAge_norm", "BonusMalus_norm")

train_loader_pair = DataLoader(train_pair, batch_size=4096, shuffle=True)
valid_loader_pair = DataLoader(valid_pair, batch_size=4096, shuffle=False)

model_pair = PairInteractionNet().to(device)
optimizer = torch.optim.Adam(model_pair.parameters(), lr=3e-3)

best_valid = float("inf")
for epoch in range(150):
    model_pair.train()
    for batch in train_loader_pair:
        var1, var2 = batch["var1"].to(device), batch["var2"].to(device)
        log_mu_glm, claim_nb = batch["log_mu_glm"].to(device), batch["claim_nb"].to(device)
        optimizer.zero_grad()
        log_lambda = model_pair(var1, var2, log_mu_glm)
        loss = poisson_deviance_loss_v2(log_lambda, claim_nb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_pair.parameters(), max_norm=1.0)
        optimizer.step()
    model_pair.eval()
    valid_losses = []
    with torch.no_grad():
        for batch in valid_loader_pair:
            var1, var2 = batch["var1"].to(device), batch["var2"].to(device)
            log_mu_glm, claim_nb = batch["log_mu_glm"].to(device), batch["claim_nb"].to(device)
            log_lambda = model_pair(var1, var2, log_mu_glm)
            valid_losses.append(poisson_deviance_loss_v2(log_lambda, claim_nb).item())
    avg_valid = sum(valid_losses) / len(valid_losses)
    best_valid = min(best_valid, avg_valid)

print(f"Meilleur valid_loss (DrivAge x BonusMalus) : {best_valid:.4f} | Référence GLM : 0.3072")

## Tentative 3 — Interaction ciblée : VehPower/VehAge/VehGas/VehBrand ✅

**Résultat retenu dans le rapport : meilleur valid_loss 0.3037-0.3043 selon le run (variabilité liée à l'initialisation aléatoire), confirmé sur test à 0.3129 contre 0.3179 pour le GLM (-1.61%).**

C'est le seul résultat ML qui bat le GLM dans ce mémoire. Plusieurs runs ont été effectués pour vérifier la stabilité du résultat avant de le retenir.

In [ ]:
from src.pricing.cann import GroupInteractionNet, GroupDataset, train_group_interaction

group_continuous_cols = ["VehPower_norm", "VehAge_norm", "VehGas_code"]

train_group = GroupDataset(train, group_continuous_cols, "VehBrand_code")
valid_group = GroupDataset(valid, group_continuous_cols, "VehBrand_code")
test_group = GroupDataset(test, group_continuous_cols, "VehBrand_code")

train_loader = DataLoader(train_group, batch_size=4096, shuffle=True)
valid_loader = DataLoader(valid_group, batch_size=4096, shuffle=False)
test_loader = DataLoader(test_group, batch_size=4096, shuffle=False)

model_group = GroupInteractionNet(n_continuous=3, brand_cardinality=11).to(device)
optimizer = torch.optim.Adam(model_group.parameters(), lr=3e-3)

model_group, best_valid, best_epoch = train_group_interaction(
    model_group, train_loader, valid_loader, n_epochs=400, optimizer=optimizer, device=device
)

In [ ]:
# Évaluation finale sur test (jamais touché pendant l'entraînement)
model_group.eval()
test_losses = []
with torch.no_grad():
    for batch in test_loader:
        continuous, brand_code = batch["continuous"].to(device), batch["brand_code"].to(device)
        log_mu_glm, claim_nb = batch["log_mu_glm"].to(device), batch["claim_nb"].to(device)
        log_lambda = model_group(continuous, brand_code, log_mu_glm)
        test_losses.append(poisson_deviance_loss_v2(log_lambda, claim_nb).item())

test_loss_group = sum(test_losses) / len(test_losses)
print("Loss sur test :", test_loss_group, "| Référence GLM test : 0.3179")

## NGBoost — sévérité distributionnelle

In [ ]:
from src.pricing.models import fit_ngboost_severity, predict_ngboost_severity

model_ngboost = fit_ngboost_severity(train_sev, n_estimators=300)
ngboost_preds = predict_ngboost_severity(model_ngboost, test_sev)

covered = (
    (test_sev["ClaimAmount_capped"].values >= ngboost_preds["pred_lower_90"].values) &
    (test_sev["ClaimAmount_capped"].values <= ngboost_preds["pred_upper_90"].values)
)
print(f"Couverture empirique intervalle 90% : {covered.mean():.2%}")  # résultat retenu : 90.58%

## Test de dépendance fréquence-sévérité (copule)

In [ ]:
from scipy.stats import spearmanr

test_sev["pred_severity"] = predict_severity(model_gamma, test_sev)
test_sev_freq = predict_frequency(model_glm_train, test_sev)

corr, pval = spearmanr(test_sev_freq, test_sev["ClaimAmount_capped"])
print(f"Corrélation de Spearman : {corr:.4f} (p={pval:.4f})")
# Résultat retenu : rho=0.078, jugé négligeable -> copule non implémentée

## Sauvegarde des modèles retenus

Point d'entrée pour `03b_pricing_cann_final.ipynb` — exécuter une seule fois.

In [ ]:
import joblib

Path("../models").mkdir(exist_ok=True)

joblib.dump(model_glm_train, "../models/glm_poisson.pkl")
joblib.dump(model_gamma, "../models/glm_gamma.pkl")
joblib.dump(model_ngboost, "../models/ngboost_severity.pkl")
torch.save(model_group.state_dict(), "../models/cann_group_interaction.pt")

print("Modèles sauvegardés dans ../models/")